In [1]:
import pandas as pd

df = pd.read_csv('../data/cosmetics.csv')
print("Loaded", len(df), "products")

Loaded 1472 products


In [2]:
# Common skincare allergens - this is your knowledge base
ALLERGENS = [
    'fragrance', 'parfum', 'alcohol', 'parabens', 'methylparaben',
    'propylparaben', 'formaldehyde', 'lanolin', 'nickel', 'gluten',
    'sulfates', 'sodium lauryl sulfate', 'sls', 'phthalates',
    'oxybenzone', 'retinol', 'salicylic acid', 'glycolic acid',
    'linalool', 'limonene', 'geraniol', 'citronellol', 'benzyl salicylate'
]

print(f"Tracking {len(ALLERGENS)} allergens")

Tracking 23 allergens


In [3]:
def parse_ingredients(ingredient_string):
    """Split raw ingredient string into a clean list"""
    if pd.isna(ingredient_string):
        return []
    ingredients = [i.strip().lower() for i in ingredient_string.split(',')]
    return ingredients

def check_allergens(ingredient_string, user_allergens):
    """Check which of the user's allergens are in a product"""
    ingredients = parse_ingredients(ingredient_string)
    found = []
    for allergen in user_allergens:
        for ingredient in ingredients:
            if allergen.lower() in ingredient:
                found.append(allergen)
                break
    return found

In [4]:
# Simulate a user who is allergic to fragrance and alcohol
user_allergens = ['fragrance', 'alcohol', 'linalool']

# Test on the first product
product = df.iloc[0]
print("Product:", product['Name'])
print("Brand:", product['Brand'])
print()

allergens_found = check_allergens(product['Ingredients'], user_allergens)

if allergens_found:
    print("WARNING - contains your allergens:", allergens_found)
else:
    print("Safe for you - no allergens found")

Product: Crème de la Mer
Brand: LA MER

WARNING - contains your allergens: ['fragrance', 'alcohol', 'linalool']


In [5]:
# Flag every product for this user's allergens
df['allergens_found'] = df['Ingredients'].apply(
    lambda x: check_allergens(x, user_allergens)
)
df['is_safe'] = df['allergens_found'].apply(lambda x: len(x) == 0)

print(f"Total products: {len(df)}")
print(f"Safe for user: {df['is_safe'].sum()}")
print(f"Contains allergens: {(~df['is_safe']).sum()}")

Total products: 1472
Safe for user: 565
Contains allergens: 907


In [6]:
safe_products = df[df['is_safe']][['Name', 'Brand', 'Label', 'Price']].head(10)
print(safe_products.to_string())

                                                           Name           Brand        Label  Price
1                                      Facial Treatment Essence           SK-II  Moisturizer    179
4                 Your Skin But Better™ CC+™ Cream with SPF 50+    IT COSMETICS  Moisturizer     38
7                               Virgin Marula Luxury Facial Oil  DRUNK ELEPHANT  Moisturizer     72
13                                      Luna Sleeping Night Oil    SUNDAY RILEY  Moisturizer    105
17            Moisture Surge 72-Hour Auto-Replenishing Hydrator        CLINIQUE  Moisturizer     39
21  COMPLEXION RESCUE™ Tinted Moisturizer Broad Spectrum SPF 30    BAREMINERALS  Moisturizer     30
26                         Virgin Marula Luxury Facial Oil Mini  DRUNK ELEPHANT  Moisturizer     40
32                 Sheer Transformation® Perfecting Moisturizer    OLEHENRIKSEN  Moisturizer     38
33                                   100 percent Pure Argan Oil     JOSIE MARAN  Moisturizer     48


In [7]:
df.to_csv('../data/products_with_allergens.csv', index=False)
print("Saved to data/products_with_allergens.csv")

Saved to data/products_with_allergens.csv


In [8]:
# Ingredients scored by effect on each skin type
# 1 = beneficial, 0 = neutral, -1 = harmful

INGREDIENT_SCORES = {
    # Hydrators - good for dry skin
    'glycerin': {'dry': 1, 'oily': 0, 'sensitive': 1, 'combination': 0, 'normal': 0},
    'hyaluronic acid': {'dry': 1, 'oily': 1, 'sensitive': 1, 'combination': 1, 'normal': 1},
    'ceramide': {'dry': 1, 'oily': 0, 'sensitive': 1, 'combination': 0, 'normal': 1},
    'shea butter': {'dry': 1, 'oily': -1, 'sensitive': 0, 'combination': -1, 'normal': 0},
    'squalane': {'dry': 1, 'oily': 0, 'sensitive': 1, 'combination': 0, 'normal': 1},

    # Actives - good for treatment
    'niacinamide': {'dry': 1, 'oily': 1, 'sensitive': 1, 'combination': 1, 'normal': 1},
    'retinol': {'dry': 0, 'oily': 1, 'sensitive': -1, 'combination': 1, 'normal': 0},
    'salicylic acid': {'dry': -1, 'oily': 1, 'sensitive': -1, 'combination': 1, 'normal': 0},
    'glycolic acid': {'dry': -1, 'oily': 1, 'sensitive': -1, 'combination': 0, 'normal': 0},
    'vitamin c': {'dry': 1, 'oily': 1, 'sensitive': 0, 'combination': 1, 'normal': 1},

    # Oils - depends on skin type
    'mineral oil': {'dry': 1, 'oily': -1, 'sensitive': 0, 'combination': -1, 'normal': 0},
    'jojoba': {'dry': 1, 'oily': 0, 'sensitive': 1, 'combination': 0, 'normal': 1},
    'coconut oil': {'dry': 1, 'oily': -1, 'sensitive': -1, 'combination': -1, 'normal': 0},

    # Irritants - bad for sensitive
    'fragrance': {'dry': 0, 'oily': 0, 'sensitive': -1, 'combination': 0, 'normal': 0},
    'alcohol denat': {'dry': -1, 'oily': 0, 'sensitive': -1, 'combination': 0, 'normal': 0},
    'sulfate': {'dry': -1, 'oily': 0, 'sensitive': -1, 'combination': 0, 'normal': 0},
}

print(f"Scoring {len(INGREDIENT_SCORES)} key ingredients")

Scoring 16 key ingredients


In [9]:
def score_product_for_skin(ingredient_string, skin_type):
    """
    Score a product for a given skin type
    Returns a score between -10 and 10
    Positive = good match, Negative = bad match
    """
    ingredients = parse_ingredients(ingredient_string)
    
    score = 0
    matched = []
    
    for ingredient in ingredients:
        for key, scores in INGREDIENT_SCORES.items():
            if key in ingredient:
                skin_score = scores.get(skin_type, 0)
                score += skin_score
                if skin_score != 0:
                    matched.append((key, skin_score))
    
    return score, matched

# Test it
skin_type = 'oily'
product = df.iloc[2]  # Drunk Elephant product

score, matched = score_product_for_skin(product['Ingredients'], skin_type)

print(f"Product: {product['Name']}")
print(f"Skin type: {skin_type}")
print(f"Score: {score}")
print(f"Matched ingredients: {matched}")

Product: Protini™ Polypeptide Cream
Skin type: oily
Score: 1
Matched ingredients: [('glycolic acid', 1)]


In [10]:
# Score all products for oily skin
df['score_oily'] = df['Ingredients'].apply(
    lambda x: score_product_for_skin(x, 'oily')[0]
)
df['score_dry'] = df['Ingredients'].apply(
    lambda x: score_product_for_skin(x, 'dry')[0]
)
df['score_sensitive'] = df['Ingredients'].apply(
    lambda x: score_product_for_skin(x, 'sensitive')[0]
)

print("Scores added for oily, dry, sensitive skin types")
print(df[['Name', 'score_oily', 'score_dry', 'score_sensitive']].head(10).to_string())

Scores added for oily, dry, sensitive skin types
                                                  Name  score_oily  score_dry  score_sensitive
0                                      Crème de la Mer          -1          0               -2
1                             Facial Treatment Essence           0          0                0
2                           Protini™ Polypeptide Cream           1          0                0
3                          The Moisturizing Soft Cream          -1          2                0
4        Your Skin But Better™ CC+™ Cream with SPF 50+           2          3                3
5                                      The Water Cream           0          3                2
6                            Lala Retro™ Whipped Cream           0          2                2
7                      Virgin Marula Luxury Facial Oil           0          0                0
8                                   Ultra Facial Cream           0          2                2
9

In [11]:
def get_top_products(skin_type, category=None, max_price=None, top_n=5):
    """
    Return top N products for a skin type
    Optionally filter by category and price
    """
    results = df[df['is_safe']].copy()
    
    # Optional filters
    if category:
        results = results[results['Label'] == category]
    if max_price:
        results = results[results['Price'] <= max_price]
    
    # Sort by skin type score
    score_col = f'score_{skin_type}'
    results = results.sort_values(score_col, ascending=False)
    
    return results[['Name', 'Brand', 'Label', 'Price', score_col]].head(top_n)

# Test it - top moisturizers for oily skin under $50
results = get_top_products(
    skin_type='oily',
    category='Moisturizer',
    max_price=50,
    top_n=5
)
print("Top moisturizers for oily skin under $50:")
print(results.to_string())

Top moisturizers for oily skin under $50:
                                                          Name         Brand        Label  Price  score_oily
4                Your Skin But Better™ CC+™ Cream with SPF 50+  IT COSMETICS  Moisturizer     38           2
34   Your Skin But Better CC+ Cream Oil-Free Matte with SPF 40  IT COSMETICS  Moisturizer     38           2
70   Your Skin But Better™ CC+Illumination™ Cream with SPF 50+  IT COSMETICS  Moisturizer     38           2
133                     Miracle Water 3-in-1 Micellar Cleanser  IT COSMETICS  Moisturizer     38           2
124             Bye Bye Redness™ Neutralizing Correcting Cream  IT COSMETICS  Moisturizer     32           2


In [12]:
df.to_csv('../data/products_scored.csv', index=False)
print("Saved to data/products_scored.csv")
print(f"Dataset shape: {df.shape}")

Saved to data/products_scored.csv
Dataset shape: (1472, 16)


In [13]:
# Top cleansers for sensitive skin under $30
print("=== Top cleansers for sensitive skin under $30 ===")
print(get_top_products('sensitive', category='Cleanser', max_price=30).to_string())

print()

# Top treatments for dry skin - no price limit
print("=== Top treatments for dry skin ===")
print(get_top_products('dry', category='Treatment', top_n=5).to_string())

=== Top cleansers for sensitive skin under $30 ===
                                                                      Name         Brand     Label  Price  score_sensitive
391  Confidence in a Cleanser™ Skin-Transforming Hydrating Cleansing Serum  IT COSMETICS  Cleanser     28                6
376                                     GinZing™ Refreshing Scrub Cleanser       ORIGINS  Cleanser     20                3
318                                                Purifying Cleansing Gel        BOSCIA  Cleanser     28                3
504                             BUBBLESHEET™ Oxygenating Deep Cleanse Mask      GLAMGLOW  Cleanser      9                3
360                              The Microdelivery Exfoliating Facial Wash    PHILOSOPHY  Cleanser      8                2

=== Top treatments for dry skin ===
                                       Name           Brand      Label  Price  score_dry
806            Clear Complexion Moisturizer          BOSCIA  Treatment     36        